# Autocorrelation Diagnostics

Regression inference assumes independent errors. Time series often violate that assumption because nearby observations can have related errors.

First-order autocorrelation is written as

$$\epsilon_t = \phi \epsilon_{t-1} + a_t,$$

where $a_t$ is independent noise. If $\phi > 0$, positive residuals tend to be followed by positive residuals and negative by negative. If $\phi < 0$, residual signs tend to alternate.

The lecture uses three diagnostics:

1. residuals plotted against time;
2. signs and runs of time-ordered residuals;
3. Durbin-Watson statistic.

The Durbin-Watson statistic is

$$d = \frac{\sum_{t=2}^{T}(e_t-e_{t-1})^2}{\sum_{t=1}^{T}e_t^2}.$$

Values near 2 suggest little first-order autocorrelation; values below 2 suggest positive autocorrelation; values above 2 suggest negative autocorrelation. Formal textbook decisions compare $d$ with lower and upper critical values and may be inconclusive.

In [ ]:
from lite_setup import ensure_packages
await ensure_packages()

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.formula.api as smf
from statsmodels.stats.stattools import durbin_watson
from checks import check_columns, check_no_missing

plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.grid'] = True
demand = pd.read_csv(Path('data/autocorrelated_demand.csv'))
print(check_columns(demand, ['Period', 'Load', 'Demand']))
model = smf.ols('Demand ~ Period + Load', data=demand).fit()
resid = model.resid
print(model.summary().tables[1])

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(demand['Period'], demand['Demand'], marker='o')
axes[0].set_ylabel('Demand')
axes[0].set_title('Observed demand')
axes[1].plot(demand['Period'], resid, marker='o')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xlabel('Period')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residuals over time')
plt.tight_layout()

In [ ]:
def run_summary(values):
    signs = np.where(values >= 0, '+', '-')
    runs = 1 + np.sum(signs[1:] != signs[:-1])
    return ''.join(signs), int(runs)

sign_string, runs = run_summary(resid.to_numpy())
print('Residual signs:')
print(sign_string)
print(f'Number of runs: {runs}')
print('Few long runs suggest positive autocorrelation; many short alternating runs suggest negative autocorrelation.')

In [ ]:
dw = durbin_watson(resid)
print(f'Durbin-Watson statistic: {dw:.3f}')
print('Rule of thumb: values near 2 suggest weak autocorrelation; values well below 2 suggest positive autocorrelation; values well above 2 suggest negative autocorrelation.')

Durbin-Watson formal decisions use lower and upper critical values from a table and can be inconclusive. In Python workflows, the statistic is often used together with the residual time plot and the practical context.

Important: autocorrelation does not necessarily make point forecasts useless, but it can make ordinary regression standard errors and intervals misleading.